<a href="https://colab.research.google.com/github/SyameimaruKoa/wd14-tagger-xmp/blob/main/run_colab_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WD14 Tagger Universal - Google Colab サーバー

このノートブックは、Google Colab 上で WD14 Tagger をサーバーモードとして起動し、ローカル環境から接続して画像タグ付けを行うためのものじゃ。
接続のセキュリティ確保と簡便化のために **Tailscale** を使用するぞ。

## 事前準備（Tailscale 認証キーの作成と Colab への登録）

### 1. Tailscale の認証キー（Auth Key）作成
1. Tailscale の管理画面の [Settings > Keys](https://login.tailscale.com/admin/settings/keys) にアクセスするのじゃ。
2. **Generate auth key** ボタンをクリックするのじゃ。
3. 設定項目を以下のように構成するのじゃ：
   - **Description**: 分かりやすい名前を入力（例: `colab-wd14-server`）
   - **Reusable**: チェックを入れる（ON）。これにより、Colab インスタンスの再起動時にも同じキーを使い回せるぞ。
   - **Ephemeral**: チェックを入れる（ON）。これをしておくと、Colab インスタンスがシャットダウンしてオフラインになった際に、Tailscale 側で自動的にデバイス（ノード）が削除（切断）されるため、ゴミが残らず綺麗じゃ。
   - **Tags**: `tag:Guest` を選択するのじゃ。
     - ※もし `tag:Guest` が選択肢に表示されない場合は、事前に Tailscale の Access Control Policy (ACL) にて `tagOwners` などで `tag:Guest` を定義しておく必要があるぞ。
4. **Generate key** をクリックし、生成された `tskey-auth-...` で始まるキーをコピーするのじゃ。

### 2. Google Colab へのシークレットの登録
1. 左側のサイドバーにある **鍵マーク（Secrets / シークレット）** アイコンをクリックするのじゃ。
2. **Add new secret** をクリックして以下を入力するのじゃ：
   - Name: `TAILSCALE_AUTHKEY`
   - Value: 先ほどコピーした Tailscale の認証キー
3. 追加したシークレットの **Notebook access (ノートブック of access)** トグルスイッチを **ON** にするのじゃ（これでコードからシークレットを読み込めるようになるぞ）。

※もしシークレットを登録していない（または登録したくない）場合は、セル実行時に入力プロンプトが表示されるので、そこにコピーしたキー、または Tailscale 画面に表示されるインストール/セットアップ用コマンド（`curl ... && sudo tailscale up --auth-key=tskey-auth-xxxx`）をそのまま貼り付ければ、キー部分を自動抽出して使用できるぞ。

## 1. 環境構築 & リポジトリの準備

プロジェクトのファイルをクローンし、必要なライブラリおよびGPU用ONNX Runtimeをインストールするのじゃ。

In [ ]:
import os
if not os.path.exists("embed_tags_universal.py"):
    !git clone https://github.com/SyameimaruKoa/wd14-tagger-xmp.git
    %cd wd14-tagger-xmp
else:
    !git pull
!pip install -r requirements.txt onnxruntime-gpu

## 2. Tailscale のインストールと接続

Tailscale をインストールして起動し、`TAILSCALE_AUTHKEY` を用いてログインするのじゃ。ホスト名は `wd14-tagger-colab` で固定されるぞ。

In [ ]:
import os, subprocess, re, time
auth = None
try:
    from google.colab import userdata
    auth = userdata.get("TAILSCALE_AUTHKEY")
except Exception:
    pass
if not auth:
    inp = input("Tailscale Auth Key (or setup command): ").strip()
    match = re.search(r"(tskey-(?:auth-)?[a-zA-Z0-9_-]+)", inp)
    auth = match.group(1) if match else inp
if not auth:
    raise ValueError("Tailscale Auth Key is required.")
!curl -fsSL https://tailscale.com/install.sh | sh
!mkdir -p /dev/net
!mknod /dev/net/tun c 10 200
!chmod 600 /dev/net/tun
subprocess.Popen(["tailscaled"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(20):
    if os.path.exists("/var/run/tailscale/tailscaled.sock"): break
    time.sleep(0.5)
res = subprocess.run(["tailscale", "up", f"--authkey={auth}", "--hostname=wd14-tagger-colab", "--accept-dns=false", "--advertise-tags=tag:Guest", "--ephemeral"], capture_output=True, text=True)
if res.returncode != 0:
    print(f"Warning: {res.stderr.strip()}")
    res = subprocess.run(["tailscale", "up", f"--authkey={auth}", "--hostname=wd14-tagger-colab", "--accept-dns=false", "--ephemeral"], capture_output=True, text=True)
    if res.returncode != 0:
        print(f"Error: {res.stderr.strip()}")
        res.check_returncode()

## 3. 推論サーバーの起動

割り当てられた Tailscale IP アドレスと MagicDNS ホスト名を表示し、サーバーをポート 5000 で起動するのじゃ。ローカル側では MagicDNS ホスト名（または表示された IP アドレス）を指定して処理を投げるのじゃ。

In [ ]:
import subprocess
ip = subprocess.run(["tailscale", "ip", "-4"], capture_output=True, text=True).stdout.strip()
print(f"Tailscale IP: {ip}")
print(f"MagicDNS Hostname: wd14-tagger-colab")
print(f"Command:\n.\\run_tagger.ps1 -Client -HostIP \"wd14-tagger-colab\" -Path \"C:\\Images\" -Organize")
!python embed_tags_universal.py --mode server --port 5000 --gpu